# Large-Scale EV Smart Charging: A Four-Pillar AI Project
## IEOR E4010 — Artificial Intelligence for OR and Financial Engineering
### Columbia University · Spring 2026 · Final Project

---

## Project Overview

You are the fleet operations manager at a delivery company depot. Every evening, **20 electric vans**
return from their delivery routes and must be plugged in overnight. Each van needs to reach at least
**90% battery charge** before departing for the next morning's routes.

The catch: your depot only has a **150 kW transformer** — but if you charged all 20 vans at full
speed simultaneously, you would need 20 × 11 kW = **220 kW**. You must coordinate charging to
stay within the transformer limit, and electricity prices vary dramatically throughout the day
(overnight cheap, midday solar dip, evening peak up to $100+/MWh).

This project builds four AI systems to solve this problem:

| Pillar | Module | What It Does |
|--------|--------|--------------|
| **1 — Machine Learning** | `ml_forecaster.py` | XGBoost predicts next-hour electricity prices from tabular features |
| **2 — Deep Learning** | `dl_forecaster.py` | LSTM learns price patterns directly from raw price sequences |
| **3 — Reinforcement Learning** | `rl_agent.py` | PPO agent learns a real-time charging policy through simulation |
| **4 — Agentic AI** | `agentic_ai.py` | GPT-4o orchestrator with tool-calling ties all modules together |

**How to use this notebook:**
1. Run cells top-to-bottom
2. Sections marked **★ STUDENT TODO** are where you implement key components
3. All code runs as-is with working baselines — your task is to improve them
4. Save your trained model files for submission (instructions at the end)


## Colab Setup

Run the cells below **once** at the start of each Colab session to clone the repository
and install dependencies. Skip this section if you are running locally.

> **Tip:** After cloning, set `SAVE_DIR` to your Google Drive path so model files
> persist across sessions.


In [ ]:
# Step 1: Clone the course repository (run once per session)
# Replace with the actual GitHub URL provided by your instructor
import os

REPO_URL = "https://github.com/COURSE_REPO/Smart_EV_Charging.git"  # TODO: instructor fills this in

if not os.path.exists("Smart_EV_Charging"):
    !git clone {REPO_URL}
    %cd Smart_EV_Charging
    print("Repository cloned successfully.")
else:
    %cd Smart_EV_Charging
    !git pull origin main
    print("Repository updated.")


In [ ]:
# Step 2: Install Python dependencies
!pip install -q -r requirements.txt
print("Dependencies installed.")


In [ ]:
# Step 3 (Optional): Mount Google Drive to save models across sessions
# Uncomment and run this cell if you want to save your model files persistently.

# from google.colab import drive
# drive.mount('/content/drive')
#
# SAVE_DIR = '/content/drive/MyDrive/ev_charging_submission'
# import os; os.makedirs(SAVE_DIR, exist_ok=True)
# print(f"Save directory: {SAVE_DIR}")

# If running locally, models save in the current directory by default.
SAVE_DIR = "."  # Change to your Drive path if using Colab + Drive


## 0. Setup & Configuration

In [ ]:
# If running LOCALLY (not Colab), uncomment the line below to install dependencies:
# !pip install -r requirements.txt


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Project modules
from config import Config, DEFAULT_CONFIG
from data_utils import (
    generate_synthetic_prices,
    load_realistic_prices,
    get_daily_price_curve,
    generate_ev_schedules,
    train_test_split_prices,
    plot_price_profile,
    plot_ev_schedules,
)

# Set random seed for reproducibility
cfg = Config()
np.random.seed(cfg.seed)
print(cfg.summary())

## 1. Data Generation

### 1a. Electricity Prices

#### The Data We Provide

We generate **365 days of synthetic hourly electricity prices** calibrated to California's CAISO
(California Independent System Operator) wholesale market patterns. Each price represents the
marginal cost of electricity in $/MWh for one hour.

**CAISO-style price structure:**

| Period | Typical Range | Why |
|--------|--------------|-----|
| Overnight (11 PM – 5 AM) | $15–30/MWh | Low demand; baseload generation excess |
| Morning peak (6–9 AM) | $40–80/MWh | Commuter demand, industrial startup |
| Solar dip (10 AM – 2 PM) | $10–25/MWh | California's "duck curve" — solar overproduction |
| Evening peak (5–9 PM) | $60–120/MWh | Solar drops off, demand surges, gas peakers fire |

The synthetic generator adds:
- **Seasonal variation**: higher prices in summer (AC load) and winter (heating)
- **Weekday/weekend patterns**: lower demand on weekends (~10% lower prices)
- **Gaussian noise**: ±15% random variation around the mean
- **Rare price spikes**: ~5% chance of extreme events ($150–300/MWh)
- **Occasional negative prices**: ~2% chance (excess renewable generation)

> **Note:** For real-world data from actual US grid operators (CAISO, ERCOT, PJM, etc.),
> uncomment Option B below. This requires `pip install gridstatus` and an internet connection.

We provide **two options** for price data:
- **Option A (default):** Synthetic CAISO-style prices — no internet needed, fully reproducible
- **Option B (optional):** Real wholesale prices from US ISOs via `gridstatus`


In [ ]:
# ==============================================================
# Option A: Synthetic prices (default — works fully offline)
# ==============================================================
prices_df = generate_synthetic_prices(cfg)
print(f"Generated {len(prices_df)} hourly price records over {cfg.price.num_days_history} days")
print(f"Price stats: mean=${prices_df['price_mwh'].mean():.1f}, "
      f"min=${prices_df['price_mwh'].min():.1f}, "
      f"max=${prices_df['price_mwh'].max():.1f}/MWh")

# ==============================================================
# Option B: Realistic prices (requires internet + gridstatus)
# Uncomment below to use real CAISO data instead:
# ==============================================================
# # !pip install gridstatus
# prices_df = load_realistic_prices(
#     iso="CAISO",
#     start_date="2024-01-01",
#     end_date="2024-12-31",
#     node="TH_SP15_GEN-APND",
# )

In [ ]:
# Visualize price patterns
plot_price_profile(prices_df, num_days=10)

### 1b. EV Fleet Schedules

#### What the EV Schedule Models

Each of the 20 EVs in the fleet has a **stochastic daily schedule**:

| Parameter | Distribution | Real-World Meaning |
|-----------|-------------|-------------------|
| Arrival time | Uniform, 2 PM – 8 PM | Vans return from delivery routes at different times |
| Arrival SoC | Truncated normal, μ=35%, σ=10%, range [20%,60%] | Depends on the day's delivery distance |
| Departure time | Uniform, 5 AM – 9 AM | Morning dispatch for next day's routes |
| Target departure SoC | 90% | Operations requirement: full charge for a full day's route |
| Battery capacity | 60 kWh | Ford E-Transit commercial van specification |
| Max charge rate | 11 kW | Level 2 AC bidirectional charger |
| Max discharge rate | 11 kW | Same hardware supports V2G (Vehicle-to-Grid) |

**Vehicle-to-Grid (V2G):** Each EV can also *sell energy back to the grid* by discharging
(action = -1). This generates revenue but causes battery degradation ($0.04/kWh wear cost).
An intelligent agent learns when V2G is profitable enough to offset the degradation cost.

**The scheduling challenge:** EVs arrive and depart at different times with different energy needs.
Some arrive almost empty at 2 PM and must leave fully charged by 5 AM — tight deadline.
Others arrive half-charged at 7 PM and don't leave until 9 AM — more flexibility.
The controller must handle all 20 EVs simultaneously, sharing one 150 kW transformer.


In [ ]:
# Generate stochastic EV arrival/departure schedules
schedules = generate_ev_schedules(cfg)

print("Sample EV schedules:")
for ev in schedules[:5]:
    print(f"  {ev}")
print(f"  ... ({len(schedules)} total)")

In [ ]:
# Visualize fleet schedules (Gantt chart — color = charging urgency)
plot_ev_schedules(schedules, cfg)

### 1c. Train/Test Split & Day Extraction

#### Why Chronological Splitting Matters

We use a **chronological 80/20 split** — the first 292 days are the training set,
the last 73 days are the test set.

**Why not random split?** Electricity prices have temporal dependencies:
- Today's prices are correlated with yesterday's prices
- Random splitting would allow the model to "look into the future" during training
- This creates artificially optimistic test performance (data leakage)
- Always use chronological splits for time-series data

| Split | Days | Hourly records |
|-------|------|---------------|
| Training | 292 (80%) | ~7,008 rows |
| Test | 73 (20%) | ~1,752 rows |

The ML and LSTM models train on historical data and are evaluated on the held-out test period —
simulating real deployment where you predict tomorrow's prices using only past data.

We also extract a **single day's price curve** at 15-minute resolution (96 steps) for use
in the RL and heuristic simulations. This is the "today" scenario that the agent operates in.


In [ ]:
# Chronological train/test split (no data leakage)
train_df, test_df = train_test_split_prices(prices_df, cfg)

# Extract one day's price curve for simulation (15-min resolution)
price_curve = get_daily_price_curve(prices_df, day_index=0, cfg=cfg)
print(f"Day 0 price curve: {price_curve.shape[0]} steps, "
      f"range ${price_curve.min():.1f}–${price_curve.max():.1f}/MWh")

plt.figure(figsize=(12, 4))
steps = np.arange(len(price_curve))
hours = (cfg.time.start_hour + steps * cfg.time.dt_hours) % 24
plt.plot(hours, price_curve, 'b-', linewidth=1.5)
plt.xlabel("Hour of Day")
plt.ylabel("Price ($/MWh)")
plt.title("Day 0 Price Curve (15-min resolution)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Baseline Strategies (Heuristics)

Before applying AI, we establish baselines with **simple rule-based strategies**.
These answer: *"How well can you do without any intelligence at all?"*

These baselines are important for two reasons:
1. They quantify the **value of intelligence** — how much better does ML/RL do vs. doing nothing smart?
2. They serve as a **sanity check** — if your AI does worse than ASAP, something is wrong

#### The Three Heuristics

| Strategy | Rule | Strength | Weakness |
|----------|------|---------|---------|
| **ASAP** (As Soon As Possible) | Charge every connected EV at full power immediately | Always meets deadlines | Ignores prices — charges at expensive peak hours |
| **ALAP** (As Late As Possible) | Calculate the latest you can start and still meet the deadline, then wait | May avoid some peak prices | Deadline-sensitive; no V2G |
| **Round Robin** | Rotate which EVs get to charge each step; prioritize urgent EVs | Smooth load profile, respects transformer limit | Complex logic, may miss price windows |

> **Key insight:** ALAP tends to do better than ASAP on price because it delays charging
> toward morning hours (lower prices), but it has no concept of *which* hours are cheapest.
> This is exactly the gap that ML price forecasting and RL control can close.


In [ ]:
from heuristics import run_all_heuristics, run_heuristic
from environment import make_env, plot_episode_results

# Run all heuristics on the SAME scenario (same schedules, same prices)
heuristic_results = run_all_heuristics(
    cfg, schedules=schedules, price_curve=price_curve, verbose=True
)

In [ ]:
# Visualize the Round Robin strategy in detail
env_rr = make_env(cfg, schedules=schedules, price_curve=price_curve)
_ = run_heuristic(env_rr, strategy="round_robin")
plot_episode_results(env_rr, title="Round Robin Baseline")

## 3. LP Optimal Solution (Gold Standard)

### Why Do We Solve an LP with Perfect Foresight?

The LP optimizer solves the charging problem to **mathematical global optimality** —
but it cheats: it knows all future electricity prices, all EV arrivals and departures,
and all energy needs in advance. No real-time controller can have this information.

**We use it as the gold standard benchmark for two reasons:**
1. **Theoretical ceiling:** It gives us the best possible cost achievable on this scenario.
   Any real algorithm (ML-forecast-based, RL, heuristic) can only do *equal to or worse*.
2. **Cost of uncertainty:** The gap between LP and RL = the economic cost of not knowing
   future prices. This tells you how much a better forecaster is worth.

### The LP Formulation

We minimize total electricity cost minus V2G revenue, subject to physical constraints:

**Decision variables:**
- `p_charge[i,t]` — power drawn from grid to charge EV `i` at time step `t` (kW), ≥ 0
- `p_discharge[i,t]` — power injected into grid by EV `i` (V2G) at step `t` (kW), ≥ 0

**Objective (minimize):**
```
Σ_t Σ_i  price[t] × p_charge[i,t] × Δt / 1000    ← electricity cost ($)
   - price[t] × p_discharge[i,t] × Δt / 1000      ← V2G revenue ($)
   + degradation_cost × p_discharge[i,t] × Δt     ← battery wear ($)
```
where Δt = 0.25 hours (15-minute steps), price in $/MWh.

**Constraints:**
- **Power bounds:** 0 ≤ p_charge[i,t] ≤ 11 kW × connected[i,t]
- **SoC dynamics:** SoC[i,t+1] = SoC[i,t] + (η_c × p_charge - p_discharge/η_d) × Δt / capacity
- **SoC bounds:** 0.05 ≤ SoC[i,t] ≤ 1.0 (5% minimum, 100% maximum)
- **Departure target:** SoC[i, departure_step] ≥ 0.90 (90% target SoC)
- **Transformer limit:** Σ_i p_charge[i,t] ≤ 150 kW at every step

We solve this using `scipy.optimize.linprog` with the HiGHS solver (typically <1 second).
The LP is a **continuous relaxation** — we don't enforce integer constraints on power
(which is acceptable since Level 2 AC chargers support variable power levels).

> **Discussion question:** Why can the LP achieve lower cost than ALAP, even though ALAP
> also delays charging to later in the night?


In [ ]:
from optimizer import solve_optimal_schedule, plot_optimal_schedule

# Solve LP with V2G enabled
lp_result = solve_optimal_schedule(
    schedules, price_curve, cfg, allow_v2g=True, verbose=True
)

In [ ]:
# Visualize LP optimal schedule
plot_optimal_schedule(lp_result, price_curve, cfg)

## 4. Pillar 1: Machine Learning — Price Forecasting

### Background: Why Forecast Electricity Prices?

The LP optimizer achieves low cost because it *knows* future prices. Real-time controllers
don't have this luxury — but a price forecast is the next best thing. If we can predict
that electricity will be cheap at 2 AM and expensive at 6 PM, we can schedule charging
accordingly.

**Approach:** Train a supervised regression model to predict `price[t+1]` (next hour's
price in $/MWh) using only information available at time `t`.

### What Data the ML Model Uses

The model ingests a **feature matrix** — one row per hour — built from the price history.
There are no external inputs (no weather, no load forecasts). All features come from the
price series itself.

**The 25+ baseline features fall into six categories:**

| Category | Examples | Intuition |
|----------|---------|-----------|
| **Lag features** | `price_lag_1` (1h ago), `price_lag_24` (yesterday same hour) | Autocorrelation — recent prices correlate with next price |
| **Extended lags** | `price_lag_168` (last week same hour) | Weekly seasonality in electricity markets |
| **Rolling statistics** | `rolling_mean_6h`, `rolling_std_24h` | Recent price level and volatility regime |
| **Cyclical time** | `hour_sin`, `hour_cos`, `dow_sin`, `dow_cos` | Hour 23 and hour 0 are adjacent — sin/cos avoids the discontinuity |
| **Price changes** | `price_diff_1h`, `price_diff_24h` | Momentum signals |
| **Peak indicators** | `is_morning_peak`, `is_solar_dip`, `is_evening_peak` | Known daily price patterns as binary features |

**Target:** `target = price[t+1]` — the price in the *next* hour. This is a
**one-step-ahead prediction** task. (For planning, the model predicts autoregressively
to generate a 24-hour forecast by feeding predictions back as inputs.)

**No lookahead bias:** All features use `shift(1)` or more — features at time `t` only
use price data from time `t` and earlier. Never `t+1` or later.

### Model: XGBoost

We use **XGBoost (Extreme Gradient Boosting)** — an ensemble of decision trees trained
via gradient boosting:
- Handles tabular features well without normalization
- Fast training on CPU
- Built-in feature importance (tells you which features matter most)
- Robust to outliers (electricity price spikes)

A **Random Forest** is also trained as a secondary model for comparison.
Evaluation metrics: **MAE** (mean absolute error, $/MWh), **RMSE**, **R²**.

---

### ★ STUDENT TODO: Extend the Feature Engineering Pipeline

Your task is to **improve upon the 25+ baseline features** by adding new columns.
Start from the baseline and add your own features on top.

**Strategy:** Start with `engineer_features()` (always call this first), then add your
custom columns to the returned DataFrame. The autograder gracefully handles custom
features — any feature not present in the holdout set is simply dropped, so the
baseline performance is always preserved.

**Ideas to try:**
- Additional lags: `t-48` (2 days ago), `t-72` (3 days ago), `t-336` (2 weeks ago)
- Interaction terms: `hour_sin × price_lag_24` — captures "expensive morning vs cheap morning"
- Exponentially weighted moving average: `ewm(span=12)` — weights recent prices more
- Day-of-week × peak indicator: `dow_sin × is_evening_peak`
- Rolling min and max (not just mean/std): captures price range in a window
- Price acceleration: second-order difference `price[t] - 2×price[t-1] + price[t-2]`


In [ ]:
from ml_forecaster import (
    engineer_features,
    train_ml_models,
    evaluate_ml_models,
    get_feature_importance,
    plot_ml_results,
    plot_feature_importance,
    save_ml_model,
)

# Build baseline feature matrix (25+ features — see ml_forecaster.py for full list)
# We will extend these features in the TODO cell below.
baseline_train_feat = engineer_features(train_df, cfg)
baseline_test_feat  = engineer_features(test_df,  cfg)

feature_cols = [c for c in baseline_train_feat.columns
                if c not in {'datetime', 'price_mwh', 'target', 'date'}]
print(f"Baseline features ({len(feature_cols)}): {feature_cols[:10]} ...")
print(f"Training set: {len(baseline_train_feat)} rows")
print(f"Test set:     {len(baseline_test_feat)} rows")
print(f"\nTarget stats (train): mean=${baseline_train_feat['target'].mean():.2f}/MWh, "
      f"std=${baseline_train_feat['target'].std():.2f}")


In [ ]:
# ============================================================
# STUDENT TODO (Section 4): Extend the Feature Engineering Pipeline
# ============================================================
# INSTRUCTIONS:
#   1. Call engineer_features() first to get all 25+ baseline features
#   2. Add your new feature columns to the returned DataFrame
#   3. Drop NaN rows introduced by new lag features
#   4. Your extended features are automatically used for training below
#
# The autograder evaluates your saved model on holdout data. Any custom
# features not present in holdout data are gracefully dropped — so always
# start from the baseline to preserve a performance floor.
#
# Tips:
#   - After adding features, inspect feat_df.isnull().sum() to check for NaNs
#   - Check correlations: feat_df.corr()['target'].abs().sort_values(ascending=False)
#   - Run cell 23 (feature importance) to see which features matter most

def my_engineer_features(df, cfg):
    """Extend the baseline feature set with custom features.
    
    Always starts from engineer_features() to include all 25+ baseline
    features, then adds additional columns below.
    
    Args:
        df:  Price DataFrame with columns [datetime, price_mwh, hour, ...]
        cfg: Config object
        
    Returns:
        DataFrame with baseline + custom features and 'target' column
    """
    # Always start with baseline features (do not remove this line)
    feat_df = engineer_features(df, cfg)

    # ===== ADD YOUR FEATURES BELOW =====
    # Example 1: additional lags
    # feat_df['price_lag_48'] = feat_df['price_mwh'].shift(48)   # 2 days ago
    # feat_df['price_lag_72'] = feat_df['price_mwh'].shift(72)   # 3 days ago

    # Example 2: exponentially weighted moving average
    # feat_df['ewm_12h'] = feat_df['price_mwh'].ewm(span=12).mean().shift(1)

    # Example 3: interaction feature
    # feat_df['peak_lag24_interaction'] = feat_df['is_evening_peak'] * feat_df['price_lag_24']

    # Example 4: rolling min / max
    # feat_df['rolling_min_24h'] = feat_df['price_mwh'].rolling(24).min().shift(1)
    # feat_df['rolling_max_24h'] = feat_df['price_mwh'].rolling(24).max().shift(1)

    # Drop rows with NaN from new lag/rolling features
    feat_df = feat_df.dropna().reset_index(drop=True)

    return feat_df


# Apply your feature engineering
train_feat = my_engineer_features(train_df, cfg)
test_feat  = my_engineer_features(test_df,  cfg)

custom_feature_cols = [c for c in train_feat.columns
                       if c not in {'datetime', 'price_mwh', 'target', 'date'}
                       and c not in [c for c in baseline_train_feat.columns]]

print(f"Total features: {len([c for c in train_feat.columns if c not in {'datetime','price_mwh','target','date'}])}")
print(f"Your new custom features ({len(custom_feature_cols)}): {custom_feature_cols}")


In [ ]:
# Train ML models (XGBoost + Random Forest) on your feature set
# Hyperparameters are set in cfg.forecast (see config.py):
#   - XGBoost: n_estimators=200, max_depth=6, learning_rate=0.1
#   - Random Forest: n_estimators=200, max_depth=10
# You can modify these via cfg before calling train_ml_models().

ml_models = train_ml_models(train_feat, cfg)

# Evaluate on test set
ml_results = evaluate_ml_models(ml_models, test_feat, verbose=True)


In [ ]:
# Visualize ML forecasting results
plot_ml_results(ml_results, test_feat, num_days=7)

In [ ]:
# Feature importance analysis
imp_df = get_feature_importance(ml_models, top_n=15)
print(imp_df.to_string(index=False))
plot_feature_importance(imp_df)

In [ ]:
# ============================================================
# Save Your ML Model for Submission
# ============================================================
import os, shutil

save_path = "ml_model.pkl"
save_ml_model(ml_models, save_path)
print(f"ML model saved: {save_path}")
print(f"  Model types: {list(ml_models['models'].keys())}")
print(f"  Feature count: {len(ml_models['feature_columns'])}")

# Optionally copy to Google Drive (uncomment if using Colab + Drive)
# shutil.copy(save_path, os.path.join(SAVE_DIR, save_path))
# print(f"  Also copied to {SAVE_DIR}")


## 5. Pillar 2: Deep Learning — LSTM Sequence Forecasting

### Background: Why Use a Sequential Deep Learning Model?

The ML forecaster (XGBoost) treats each hour as an independent row with hand-crafted lag
features. This works well, but requires domain knowledge to design the right features.

An **LSTM (Long Short-Term Memory)** network learns from raw price sequences directly —
it doesn't need you to manually specify "use the price from 168 hours ago." It reads the
sequence and learns which parts matter.

### How the LSTM Processes Price Data

**Input:** A sliding window of the last **168 hours (1 week)** of raw hourly prices.
```
Sequence:  [price_{t-167}, price_{t-166}, ..., price_{t-1}, price_t]
Target:     price_{t+1}
```

**Why 168 steps?** One week captures all weekly seasonality patterns:
- Weekday vs. weekend price differences
- Weekly delivery patterns
- Recurring peak/off-peak cycles

**Z-score normalization:** All prices are normalized before feeding into the LSTM:
```
price_normalized = (price - training_mean) / training_std
```
The training set mean and std are saved alongside the model. At prediction time, the
output is denormalized: `predicted_price = output × training_std + training_mean`.
This is critical — the autograder loads these normalization stats from your saved model.

### LSTM Architecture (baseline)

```
Input:  (batch_size, 168, 1)        ← 168-step sequence, 1 feature (price)
LSTM Layer 1: hidden_size=64        ← learns temporal patterns
LSTM Layer 2: hidden_size=64        ← higher-level patterns
Dropout: 0.2                        ← regularization
FC Layer 1: 64 → 32                 ← compress to prediction
FC Layer 2: 32 → 1                  ← scalar next-hour price prediction
Output: scalar price ($/MWh), denormalized
```

**Training details:**
- Optimizer: Adam (lr=0.001)
- Loss: Mean Squared Error (MSE)
- LR scheduler: ReduceLROnPlateau (halves LR if val loss doesn't improve for 5 epochs)
- Early stopping: stops training if validation loss doesn't improve for 10 epochs
- Gradient clipping: max_norm=1.0 (prevents exploding gradients)

### 24-Hour Autoregressive Forecasting

After training, the model generates 24-hour price forecasts **autoregressively**:
1. Take the last 168 hours of actual prices as the initial context window
2. Predict hour `t+1` → append to window, drop oldest
3. Predict hour `t+2` using the updated window (now contains one predicted value)
4. Repeat for 24 steps

Note: errors compound in autoregressive forecasting. The 1-step-ahead accuracy (R²)
is better than the 24-step-ahead accuracy.

### ML vs. DL: When Does LSTM Win?

On this structured tabular price data with known seasonal patterns, **XGBoost with
good features often matches or beats LSTM**. LSTM has an advantage when:
- Patterns are too complex to manually engineer features for
- The raw sequence contains information beyond lag/rolling stats
- Multi-variate inputs are available (price + weather + grid load)

Students who add good features to XGBoost typically get better results than
a default LSTM. Students who tune the LSTM architecture often close the gap.

---

### ★ STUDENT TODO: Improve the LSTM Architecture and Training

Your task is to beat the baseline LSTM (2-layer, hidden_size=64, seq_len=168).

**Tunable parameters (Option A — easiest):**
- `lstm_hidden_size`: Try 128 or 256 (more capacity)
- `lstm_num_layers`: Try 3 or 4 (deeper network)
- `lstm_seq_len`: Try 24 (just 1 day of context) or 336 (2 weeks)
- `lstm_epochs`: Increase for longer training (with early stopping, it auto-stops)
- `lstm_dropout`: Try 0.1 (less regularization) or 0.3 (more)

**Advanced options (Option B — more involved):**
- Subclass `PriceLSTM` to modify the architecture
- Replace LSTM with GRU (`nn.GRU` instead of `nn.LSTM`) — often trains faster
- Add attention mechanism over the LSTM hidden states
- Multi-variate input: add hour-of-day as a second feature channel


In [ ]:
from dl_forecaster import (
    train_lstm,
    evaluate_lstm,
    plot_dl_results,
)

train_prices = train_df["price_mwh"].values
test_prices = test_df["price_mwh"].values

In [ ]:
# ============================================================
# STUDENT TODO (Section 5): Improve the LSTM Architecture
# ============================================================
# OPTION A: Modify config hyperparameters (recommended starting point)
# These override the defaults in config.py for this session only.

cfg.forecast.lstm_hidden_size = 64    # Baseline: 64. Try: 128, 256
cfg.forecast.lstm_num_layers  = 2     # Baseline: 2.  Try: 3, 4
cfg.forecast.lstm_dropout     = 0.2   # Baseline: 0.2. Try: 0.1, 0.3
cfg.forecast.lstm_seq_len     = 168   # Baseline: 168 (1 week). Try: 24, 48, 336
cfg.forecast.lstm_epochs      = 30    # Increase for better convergence (early stopping auto-stops)

print(f"LSTM config: hidden={cfg.forecast.lstm_hidden_size}, layers={cfg.forecast.lstm_num_layers}, "
      f"seq_len={cfg.forecast.lstm_seq_len}, dropout={cfg.forecast.lstm_dropout}, epochs={cfg.forecast.lstm_epochs}")


# OPTION B: Custom Architecture (advanced — optional)
# Uncomment the block below to define a custom LSTM class.
# Your class must accept hidden_size, num_layers, dropout as constructor args
# and implement a forward(x) method that takes (batch, seq_len, 1) tensors.

# from dl_forecaster import PriceLSTM
# import torch.nn as nn
#
# class MyImprovedLSTM(PriceLSTM):
#     """Your improved LSTM architecture."""
#
#     def __init__(self, hidden_size=128, num_layers=3, dropout=0.2):
#         super().__init__(hidden_size=hidden_size, num_layers=num_layers, dropout=dropout)
#         # Example: override the FC head with a larger network
#         # self.fc = nn.Sequential(
#         #     nn.Linear(hidden_size, 64),
#         #     nn.ReLU(),
#         #     nn.Dropout(dropout),
#         #     nn.Linear(64, 32),
#         #     nn.ReLU(),
#         #     nn.Linear(32, 1),
#         # )
#
#     def forward(self, x):
#         # x: (batch, seq_len, 1)
#         lstm_out, _ = self.lstm(x)
#         last_hidden = lstm_out[:, -1, :]   # Take last timestep
#         out = self.fc(last_hidden)
#         return out.squeeze(-1)
#
# model_class = MyImprovedLSTM  # Pass to train_lstm below
model_class = None  # Set to MyImprovedLSTM if using Option B


In [ ]:
# Train LSTM with your configured hyperparameters
# Training prints loss per epoch; early stopping will halt if val_loss plateaus.
lstm_dict = train_lstm(
    train_prices,
    val_prices=test_prices[:2000],
    cfg=cfg,
    verbose=True,
    # model_class=model_class,   # Uncomment if using Option B custom architecture
)

print(f"\nTraining complete. Best val loss at epoch {lstm_dict['best_epoch']}")
print(f"Normalization: mean=${lstm_dict['price_mean']:.2f}/MWh, std=${lstm_dict['price_std']:.2f}")


In [ ]:
# Evaluate LSTM on test set
lstm_results = evaluate_lstm(lstm_dict, test_prices)

In [ ]:
# Head-to-head comparison: ML vs DL
plot_dl_results(lstm_results, ml_results=ml_results, history=lstm_dict["history"])

# Comparison table
print("\n" + "="*55)
print("ML vs. DL Comparison")
print("="*55)
print(f"{'Model':<15} {'MAE ($/MWh)':<15} {'RMSE ($/MWh)':<15} {'R²':<10}")
print("-"*55)
for name, res in ml_results.items():
    print(f"{name:<15} ${res['mae']:<14.2f} ${res['rmse']:<14.2f} {res['r2']:<10.4f}")
print(f"{'LSTM':<15} ${lstm_results['mae']:<14.2f} ${lstm_results['rmse']:<14.2f} {lstm_results['r2']:<10.4f}")

In [ ]:
# ============================================================
# Save Your LSTM Model for Submission
# ============================================================
from dl_forecaster import save_lstm_model
import shutil, os

save_path = "lstm_model.pth"
save_lstm_model(lstm_dict, save_path)
print(f"LSTM model saved: {save_path}")
print(f"  Architecture: hidden_size={lstm_dict.get('hidden_size', cfg.forecast.lstm_hidden_size)}, "
      f"num_layers={lstm_dict.get('num_layers', cfg.forecast.lstm_num_layers)}, "
      f"seq_len={lstm_dict.get('seq_len', cfg.forecast.lstm_seq_len)}")
print(f"  Normalization stats: mean={lstm_dict['price_mean']:.3f}, std={lstm_dict['price_std']:.3f}")

# Optionally copy to Google Drive
# shutil.copy(save_path, os.path.join(SAVE_DIR, save_path))


## 6. Pillar 3: Reinforcement Learning — PPO Agent

### Background: Why Reinforcement Learning?

The LP optimizer needs to know all future prices. The ML/LSTM forecasters predict prices
but can't directly control the chargers. **Reinforcement Learning closes this gap:**
the RL agent learns a *control policy* — given the current situation, what charging actions
to take — through millions of simulated interactions with the environment.

Unlike the LP, the RL agent sees only **current information** and must learn to make
good decisions under uncertainty. It discovers price patterns implicitly from experience.

### The Gymnasium Environment

The environment (`environment.py`) wraps the charging problem as a standard RL interface:
```
obs, info = env.reset()        # Start a new episode (random EVs and prices)
action = agent.predict(obs)    # Agent chooses charging actions
obs, reward, done, _, info = env.step(action)  # Environment advances one timestep
```
Each episode = 24 hours (96 steps × 15 minutes = one overnight charging session).

---

### State Space: What the Agent Observes (63-dimensional vector)

At every time step, the agent receives a 63-dimensional observation vector:

**Per-EV features (3 × 20 = 60 values):**

| Index | Feature | Range | Meaning |
|-------|---------|-------|---------|
| `3i` | `current_soc` | [0, 1] | EV i's current battery level (0=empty, 1=full) |
| `3i+1` | `time_to_depart` | [0, 1] | 1=just arrived, 0=about to leave — urgency signal |
| `3i+2` | `is_connected` | {0, 1} | 1 if EV is plugged in at this timestep |

**Global features (3 values):**

| Index | Feature | Range | Meaning |
|-------|---------|-------|---------|
| `60` | `norm_time` | [0, 1] | Fraction of the day elapsed — correlates with price patterns |
| `61` | `norm_price` | [0, 1] | Current electricity price / max observed price |
| `62` | `load_fraction` | [0, 1] | Previous step's total power / transformer limit |

**Important:** The agent sees the **current price** (index 61) but **NOT future prices**.
It must learn from `norm_time` (index 60) that "it's 2 AM, prices are usually low now."
This is fundamentally different from the LP optimizer, which cheats with complete foresight.

---

### Action Space: What the Agent Controls (20-dimensional continuous)

One action value per EV, clipped to **[-1, +1]**:

| Value | Interpretation |
|-------|---------------|
| `+1.0` | Charge at full rate (+11 kW) |
| `+0.5` | Charge at half rate (+5.5 kW) |
| `0.0` | Idle — no power exchange |
| `-0.5` | Discharge at half rate (V2G, -5.5 kW) |
| `-1.0` | Discharge at full rate (V2G, -11 kW) |

Actions for **disconnected EVs** are automatically zeroed out by the environment.

**Transformer constraint enforcement:** If the sum of all charging actions would exceed
150 kW, the environment **proportionally scales all charging actions down** to exactly
hit the limit. This ensures physical feasibility without hard action clipping.

---

### Reward Function: The Default Design

The agent maximizes cumulative reward over the episode. The **default reward** at each step:

```
reward = -(price_weight × step_cost) - deadline_penalty - overload_penalty

where:
  step_cost = charging_cost - v2g_revenue + degradation_cost
  charging_cost  = Σ_i energy_charged[i] × price/1000        ($/MWh → $/kWh)
  v2g_revenue    = Σ_i energy_discharged[i] × price/1000
  degradation    = Σ_i energy_discharged[i] × $0.04/kWh

  deadline_penalty: fires when EV departs with SoC < 90%
                    = penalty_weight × (target_soc - actual_soc)
  overload_penalty: fires if total power > 150 kW after scaling
                    (should rarely trigger; indicates reward shaping issue)
```

The reward design has a fundamental **trade-off**:
- Setting `deadline_penalty` too low → agent learns to be cheap but misses departure targets
- Setting it too high → agent charges immediately regardless of price (like ASAP)
- The sweet spot rewards both cost efficiency AND constraint satisfaction

---

### PPO Algorithm

**PPO (Proximal Policy Optimization)** is an actor-critic RL algorithm:

- **Actor (policy network):** Maps state → Gaussian distribution over actions
  - Architecture: Linear(63→256) → ReLU → Linear(256→128) → ReLU → Linear(128→20)
  - Outputs mean and log-std for each of the 20 action dimensions
- **Critic (value network):** Maps state → expected future reward
  - Same architecture, scalar output

PPO trains by collecting 2048 environment steps (a "rollout"), then running
10 mini-batch gradient updates. The "proximal" constraint prevents the policy
from changing too drastically in one update (stabilizes training).

**Training typically requires 100k–500k timesteps** (~5-30 min on CPU, faster on GPU).
The training curve should show cost decreasing and targets-met increasing over time.

---

### ★ STUDENT TODO: Design a Better Reward Function and Tune PPO

**Part A — Reward Function (required):** Design a reward function that better balances
cost minimization with departure target satisfaction.

The function receives at every time step:
- `step_cost` (float): net electricity cost this step ($)
- `deadline_penalty` (float): departure SoC shortfall penalty (pre-computed, you can scale it)
- `overload_penalty` (float): transformer overload penalty
- `charging_energy` (array, shape 20): kWh charged per EV this step
- `discharging_energy` (array, shape 20): kWh discharged (V2G) per EV this step
- `soc_array` (array, shape 20): current SoC for all EVs [0,1]
- `cfg` (Config): full configuration object
- `current_step` (int): timestep within episode [0, 95]
- `price` (float): current electricity price ($/MWh)

Return a scalar float — higher is better for the agent.

**Part B — Hyperparameter Tuning (encouraged):** Experiment with PPO hyperparameters.
More training steps and a larger network usually improve performance, with diminishing returns.


In [ ]:
from rl_agent import (
    train_rl_agent,
    evaluate_rl_agent,
    run_single_episode,
    plot_training_curves,
    HAS_SB3,
)

if not HAS_SB3:
    print("⚠️  stable-baselines3 not installed. Skipping RL training.")
    print("   Install with: pip install stable-baselines3")

In [ ]:
# ============================================================
# STUDENT TODO (Section 6, Part A): Design Your Reward Function
# ============================================================
# The default reward is: -(price_weight * step_cost) - deadline_penalty - overload_penalty
#
# Modify the function below to design a better reward.
# Your reward function is passed to make_env() and train_rl_agent().
#
# Reward design ideas:
#   1. Increase deadline_penalty weight (the provided value may be too low to enforce constraints)
#   2. Add a "charging urgency" bonus: reward more when low-SoC EVs are near departure
#   3. Penalize abrupt power changes for smooth grid operation
#   4. V2G incentive: bonus for discharging during high price periods (price > threshold)
#   5. Progressive penalty: escalate penalty as departure approaches (time_to_depart → 0)
#
# The autograder evaluates your RL agent on cost + departure targets.
# A good reward function should learn both price awareness AND deadline satisfaction.

def my_custom_reward(
    step_cost,
    deadline_penalty,
    overload_penalty,
    charging_energy,
    discharging_energy,
    soc_array,
    cfg,
    current_step,
    price,
):
    """
    Custom reward function for the PPO charging agent.
    
    Must return a scalar float. Higher = better for the agent.
    """
    import numpy as np

    # ===== YOUR REWARD FUNCTION =====
    # Default implementation shown for reference — modify this:
    reward = (
        - cfg.rl.reward_price_weight * step_cost
        - deadline_penalty
        - overload_penalty
    )

    # Example additions (uncomment to try):

    # 1. Stronger deadline enforcement (multiply the provided penalty by a factor)
    # reward -= 2.0 * deadline_penalty   # double the deadline penalty

    # 2. V2G bonus during peak prices
    # if price > 60:   # $/MWh, adjust threshold
    #     reward += 0.3 * np.sum(discharging_energy)

    # 3. Urgency-weighted charging bonus
    # for i, ev in enumerate(cfg.fleet.num_evs):
    #     urgency = 1.0 - soc_array[i]   # low SoC = high urgency
    #     reward += 0.01 * urgency * charging_energy[i]

    return float(reward)


In [ ]:
# ============================================================
# STUDENT TODO (Section 6, Part B): Tune PPO Hyperparameters
# ============================================================
# Override RL training parameters before calling train_rl_agent().
# Start with small changes and observe the training curve response.
#
# Key parameters:
#   total_timesteps : More steps → better policy, but slower training
#                     100k steps ≈ 5 min CPU; 500k ≈ 25 min; 1M ≈ 50 min
#   learning_rate   : Step size for gradient updates
#                     Too high → unstable; too low → slow convergence
#   n_steps         : Environment steps per rollout (larger = lower variance estimates)
#   batch_size      : Mini-batch size for gradient steps (must divide n_steps)
#   gamma           : Discount factor for future rewards (higher = longer horizon)
#
# Note: policy_net_arch is set in RLConfig but passed to PPO as net_arch.
# To change network size, modify cfg.rl below and the code uses it in train_rl_agent.

if HAS_SB3:
    # ===== TUNE THESE PARAMETERS =====
    cfg.rl.total_timesteps = 100_000    # Baseline: 100k. Try: 200k, 500k
    cfg.rl.learning_rate   = 3e-4      # Baseline: 3e-4. Try: 1e-4, 1e-3
    cfg.rl.n_steps         = 2048      # Baseline: 2048. Try: 1024, 4096
    cfg.rl.batch_size      = 64        # Baseline: 64.   Try: 128, 256
    cfg.rl.gamma           = 0.99      # Baseline: 0.99. Try: 0.95, 0.995
    # cfg.rl.policy_net_arch = [512, 256]  # Baseline: [256, 128]. Try: [512, 256], [256, 256, 128]

    print("Training PPO agent with your configuration...")
    print(f"  Total timesteps: {cfg.rl.total_timesteps:,}")
    print(f"  Learning rate: {cfg.rl.learning_rate}")
    print(f"  Network arch: {getattr(cfg.rl, 'policy_net_arch', [256, 128])}")


In [ ]:
if HAS_SB3:
    # Train PPO agent with your custom reward function and hyperparameters
    rl_result = train_rl_agent(
        cfg,
        custom_reward_fn=my_custom_reward,   # Your reward function from Part A
        verbose=True,
    )
    print(f"\nTraining complete!")
    print(f"  Final mean episode cost: ${rl_result['training_stats']['episode_costs'][-1]:.2f}")


In [ ]:
if HAS_SB3:
    # Plot training progress (should see cost decreasing, targets met increasing)
    plot_training_curves(rl_result["training_stats"])

In [ ]:
if HAS_SB3:
    # Evaluate on the same scenario as heuristics/LP for fair comparison
    rl_eval = evaluate_rl_agent(
        rl_result["model"], cfg,
        schedules=schedules, price_curve=price_curve,
        n_episodes=5,
    )

    # Run single episode for detailed visualization
    rl_episode = run_single_episode(
        rl_result["model"], cfg,
        schedules=schedules, price_curve=price_curve,
    )
    plot_episode_results(rl_episode["env"], title="PPO RL Agent")

In [ ]:
# ============================================================
# Save Your RL Agent for Submission
# ============================================================
import shutil, os

if HAS_SB3:
    save_path = "rl_agent"   # stable-baselines3 automatically appends .zip
    rl_result["model"].save(save_path)
    print(f"RL agent saved: {save_path}.zip")
    print(f"  Policy architecture: {rl_result['model'].policy.net_arch}")
    print(f"  Total timesteps trained: {cfg.rl.total_timesteps:,}")

    # Optionally copy to Google Drive
    # shutil.copy(save_path + ".zip", os.path.join(SAVE_DIR, "rl_agent.zip"))
else:
    print("RL not available (stable-baselines3 not installed). Skipping save.")


## 7. Comprehensive Strategy Comparison

In [ ]:
# Collect all results into a comparison table
comparison = []

# Heuristics
for h_result in heuristic_results:
    comparison.append(h_result)

# LP Optimal
comparison.append({
    "strategy": "lp_optimal",
    "net_cost": lp_result["net_cost"],
    "total_cost": lp_result["total_cost"],
    "v2g_revenue": lp_result["v2g_revenue"],
    "degradation_cost": lp_result.get("degradation_cost", 0),
    "penalties": 0,  # LP satisfies all constraints
    "evs_meeting_target": lp_result["evs_meeting_target"],
    "total_evs": lp_result["total_evs"],
})

# RL Agent (if trained)
if HAS_SB3:
    comparison.append(rl_episode)

# Display comparison table
comp_df = pd.DataFrame(comparison)
display_cols = ["strategy", "net_cost", "v2g_revenue", "penalties",
                "evs_meeting_target", "total_evs"]
existing_cols = [c for c in display_cols if c in comp_df.columns]
print("\n" + "="*80)
print("STRATEGY COMPARISON DASHBOARD")
print("="*80)
print(comp_df[existing_cols].to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

strategies = comp_df["strategy"].values
costs = comp_df["net_cost"].values

# Panel 1: Net Cost
ax1 = axes[0]
colors = ['#94A3B8', '#94A3B8', '#94A3B8', '#0D9488', '#F59E0B'][:len(strategies)]
bars = ax1.bar(strategies, costs, color=colors)
ax1.set_ylabel("Net Cost ($)")
ax1.set_title("Net Electricity Cost by Strategy")
ax1.tick_params(axis='x', rotation=45)
for bar, cost in zip(bars, costs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"${cost:.1f}", ha='center', va='bottom', fontsize=9)
ax1.grid(True, axis='y', alpha=0.3)

# Panel 2: EVs Meeting Target
ax2 = axes[1]
targets = comp_df["evs_meeting_target"].values
total = comp_df["total_evs"].values[0]
ax2.bar(strategies, targets, color=colors)
ax2.axhline(y=total, color='red', linestyle='--', label=f'Total EVs ({total})')
ax2.set_ylabel("EVs Meeting Target")
ax2.set_title("Departure Deadline Compliance")
ax2.tick_params(axis='x', rotation=45)
ax2.legend()
ax2.grid(True, axis='y', alpha=0.3)

# Panel 3: V2G Revenue
ax3 = axes[2]
if "v2g_revenue" in comp_df.columns:
    v2g = comp_df["v2g_revenue"].values
    ax3.bar(strategies, v2g, color=colors)
    ax3.set_ylabel("V2G Revenue ($)")
    ax3.set_title("Vehicle-to-Grid Revenue")
    ax3.tick_params(axis='x', rotation=45)
    ax3.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Pillar 4: Agentic AI — LLM Orchestrator

The agentic AI uses an LLM (GPT-4o) with **tool-calling** to orchestrate all modules.
It provides a natural-language interface for fleet operators.

**Two modes:**
- **Live mode:** Set `OPENAI_API_KEY` environment variable to use real GPT-4o
- **Mock mode (default):** Works without any API key — demonstrates the full agentic pattern

To enable live mode:
```python
import os
os.environ["OPENAI_API_KEY"] = "sk-..."
```

In [ ]:
from agentic_ai import create_agent

# Get the best ML model for the agent's toolbox
best_ml_model = ml_models["models"].get("xgboost", None)

# Create the agent (auto-detects mock vs live mode based on API key)
agent = create_agent(
    cfg=cfg,
    schedules=schedules,
    price_curve=price_curve,
    ml_model=best_ml_model,
    lstm_dict=lstm_dict if 'lstm_dict' in dir() else None,
    rl_model=rl_result["model"] if HAS_SB3 and 'rl_result' in dir() else None,
    prices_df=prices_df,
)

In [ ]:
# Demo: Ask the agent about the cheapest charging plan
response = agent.chat("What's the cheapest charging plan for tonight?")
print(response)

In [ ]:
# Demo: Compare all strategies
response = agent.chat("Compare all strategies and tell me which is best.")
print(response)

In [ ]:
# Demo: V2G advice
response = agent.chat("Should I use V2G to sell energy during the evening peak?")
print(response)

## 9. Analysis Report (Student Deliverable)

### Instructions

Write your analysis **directly in this notebook** — add markdown cells and code cells below.
Your analysis should be thorough enough for a fellow graduate student unfamiliar with your
specific experiment choices to understand what you did and why.

Answer **all four questions** below. Aim for 2-4 paragraphs per question, supported by
numbers from your experiments (cite specific MAE values, cost figures, etc.).

---

### Question 1: ML vs. Deep Learning Forecasting

**Required sub-questions:**
- Which model achieved lower test-set MAE ($/MWh)? By what percentage?
- What were the top 5 most important features for your XGBoost model? Why do you think
  those features matter for electricity price prediction?
- Did your custom features improve over the baseline? Report the MAE before and after
  your feature engineering changes.
- When would you expect LSTM to outperform XGBoost on this task? When would XGBoost win?
  (Think about data size, feature engineering effort, and temporal dependency structure)
- What is the fundamental difference between how XGBoost and LSTM use historical price data?

---

### Question 2: RL vs. LP — Cost of Uncertainty

**Required sub-questions:**
- What is the **optimality gap** between your RL agent and the LP? Express as a percentage:
  `gap = (RL_cost - LP_cost) / LP_cost × 100%`
- How many EVs out of 20 met their departure target (≥90% SoC) for each strategy?
  Fill in the table below:

| Strategy | Net Cost ($) | EVs Meeting Target (out of 20) |
|----------|-------------|-------------------------------|
| ASAP | | |
| ALAP | | |
| Round Robin | | |
| LP Optimal | | |
| Your RL Agent | | |

- Why can the LP achieve lower cost than even a well-trained RL agent?
- If you had unlimited compute, what would happen to the optimality gap as you trained
  the RL agent for 10× more timesteps?
- Does V2G improve total cost for the LP? By approximately how much?

---

### Question 3: Reward Function Design

**Required sub-questions:**
- What reward function design did you implement? Explain your design choices.
- How did changing the reward function affect the training curve?
  (Show or describe how episode cost and targets-met evolved during training)
- What is the fundamental tension in reward design for this problem? How did you
  balance cost efficiency vs. constraint satisfaction?
- Describe one reward design you tried that *didn't* work well, and explain why.
- If you added V2G revenue into the reward, how did this change agent behavior?

---

### Question 4: Deployment Recommendation

**Required sub-questions:**
- If you were a fleet operations manager, which strategy would you recommend deploying
  and why? (Consider reliability, computational cost, and interpretability)
- What additional data sources (beyond electricity prices) would most improve the system's
  performance? (Weather? Traffic? Driver behavior?)
- What risks or failure modes do you see in deploying an RL agent for real EV charging?
  How would you mitigate them?
- If electricity prices became highly volatile (e.g., negative prices 20% of the time),
  which approach would degrade the least? Why?


In [ ]:
# ============================================================
# YOUR ANALYSIS CELLS START HERE
# ============================================================
# Add markdown cells for text discussion and code cells for plots/calculations.
# Example: Compute and report the optimality gap

if 'lp_result' in dir() and 'rl_eval' in dir():
    lp_cost = lp_result.get('net_cost', float('nan'))
    rl_cost = rl_eval.get('net_cost_mean', float('nan'))
    gap_pct = (rl_cost - lp_cost) / abs(lp_cost) * 100 if lp_cost != 0 else float('nan')
    print(f"LP Optimal cost:   ${lp_cost:.2f}")
    print(f"RL Agent cost:     ${rl_cost:.2f}")
    print(f"Optimality gap:    {gap_pct:.1f}%")
    print(f"\nThis gap represents the economic cost of not having perfect price foresight.")


## 10. Submission

Run the cell below to package your model files for submission.
Upload the generated `.zip` file to Courseworks.


In [ ]:
# ============================================================
# Prepare Submission Package
# ============================================================
import json, os, shutil, zipfile

# ===== FILL IN YOUR INFORMATION =====
STUDENT_NAME = "Your Full Name"    # TODO: Replace with your name
STUDENT_UNI  = "your_uni"          # TODO: Replace with your UNI (e.g., "js1234")

if STUDENT_UNI == "your_uni":
    print("ERROR: Please set STUDENT_NAME and STUDENT_UNI before running this cell!")
else:
    # Create submission folder
    submission_dir = f"submission_{STUDENT_UNI}"
    os.makedirs(submission_dir, exist_ok=True)

    # Copy model files
    required_files = ["ml_model.pkl", "lstm_model.pth", "rl_agent.zip"]
    found = []
    missing = []
    for fname in required_files:
        if os.path.exists(fname):
            shutil.copy(fname, os.path.join(submission_dir, fname))
            found.append(fname)
        else:
            missing.append(fname)

    # Write student info
    info = {"name": STUDENT_NAME, "uni": STUDENT_UNI}
    with open(os.path.join(submission_dir, "student_info.json"), "w") as f:
        json.dump(info, f, indent=2)

    # Zip the folder
    zip_path = f"{submission_dir}.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fname in os.listdir(submission_dir):
            zf.write(os.path.join(submission_dir, fname), fname)

    print(f"Submission package: {zip_path}")
    print(f"  Student: {STUDENT_NAME} ({STUDENT_UNI})")
    print(f"  Files included: {found}")
    if missing:
        print(f"  WARNING — Missing files: {missing}")
        print("  Train and save those models before submitting!")
    else:
        print(f"  All required files present. You are ready to submit!")
    print(f"\nUpload '{zip_path}' to Courseworks.")

    # Optionally copy to Google Drive
    # shutil.copy(zip_path, os.path.join(SAVE_DIR, zip_path))
